### **Section 1: Data Selection and Sampling**

#### **Problem Definition: A Digital Twin of My Life at Minerva**
Since I began my journey as a full-time student at Minerva University, my university Gmail account has essentially served as the central operating system for my entire life. This single account is far more than just an inbox for class assignments. Because the Minerva experience is so immersive, this account acts as the primary contact point for my professional growth, my academic responsibilities, and my personal scheduling. Every meeting, internship interview, class session, and social reminder is recorded here. Over time, my calendar has become a digital twin of my undergraduate experience. It provides a comprehensive archive of exactly how I have spent my time over the last few years.

The objective of this project is to explore whether a machine learning model can successfully categorize the different spheres of my life based on the language and metadata found in my schedule. I want to see if the patterns in my event entries are distinct enough to allow a model to accurately distinguish between Academic, Professional, and Personal activities. This is more than just a simple classification task. It is an attempt to see if an algorithm can recognize the different identities I project into my digital calendar.

<div style="text-align: center;">
  <img src="January_calendar.png" alt="Plot" width="700">
  <p><em>Figure 1. A snapshot of my Google Calendar from January 2026 showing the many events I am scheduled for and why this data is rich enough for my analysis.</em></p>
</div>



#### **Data Acquisition and Feature Focus**
I obtained my data by exporting my primary archive directly from Google Calendar in the iCalendar (.ics) format. While the download itself was a simple process, the file structure I received was incredibly dense and machine-heavy. A Google Calendar export is a collection of VEVENT blocks that contain a massive amount of metadata generated by the Google system. It is important to note that a standard .ics file contains dozens of hidden fields that were not relevant to my goals. 

For this project, I specifically focused on extracting a set of ten core features that I believed would provide the most signal for my models. These focused features are:
*   **summary:** The title of the event, which serves as my primary text feature.
*   **dtstart and dtend:** The precise timestamps for the start and completion of each activity.
*   **duration:** The length of the event in minutes.
*   **location:** The physical address or virtual meeting link.
*   **timezone and timezone_name:** The human-readable and technical identifiers for the event's time zone.
*   **utc_offset:** The numeric offset from UTC, which is vital for aligning events globally.
*   **is_dst:** A binary indicator showing whether daylight saving time was active.
*   **Label:** The target category I manually assigned to each event.

#### **The Technical Challenge of Data Ingestion**
While obtaining the raw file was easy, turning it into a structured Excel spreadsheet with these specific features was my biggest obstacle. I spent about two days struggling to parse the .ics data manually because I kept hitting technical errors. The most frustrating issue was that standard calendar parsers created timezone-aware objects. When I tried to save these objects into an Excel file using the Python pandas library, the code crashed because Excel cannot store those specific timestamp formats directly. 

I also had to deal with a quirk of the .ics format known as folded lines. In these files, long text strings are arbitrarily split across multiple rows with indentation. This breaks basic parsing logic because it cuts words in half and confuses regular expressions. To solve these problems, I developed a specialized Python script with assistance from ChatGPT. This script unfolded the lines before parsing and converted the technical timestamps into plain strings to avoid Excel errors. It also calculated missing durations for events that only had start and end times. This engineering work was necessary to transform messy raw files into the structured, ten-column spreadsheet I needed for modeling.

#### **Manual Labeling and Consistency Rules**
Once the data was clean, I manually labeled all 1,228 events in the dataset. This was a labor-intensive process, but it allowed me to understand the nuances of my data. I encountered several data points that could logically fit into more than one category, so I defined strict rules to maintain consistency. 

First, I classified everything related to my CP192 TA work as academic. One could argue this is professional since it is a work-study job, but I chose academic because the tasks are directly tied to the university curriculum. Second, I labeled anything related to my Cornerstone Civic Projects or other civic work as academic. Even though I gained work experience from them, I viewed them as part of my school requirements. Finally, I put all CTD meetings in the professional category. These meetings were specifically focused on helping me with my future career, so they felt distinct from my coursework. I maintained these exact definitions for every edge case to ensure my labels remained consistent.

#### **Sampling Bias and Data Distribution**
This labeling process helped me recognize a clear sampling bias in my archive. Because this is a university-affiliated account, my life looks much more Professional and Academic on paper than it actually is. Spontaneous personal events, such as quick meals with friends, rarely make it onto my calendar. In contrast, every single class or formal meeting is recorded perfectly. This means the model is not learning about my life in its entirety. Instead, it is learning about my habits as a scheduler. 

**The final counts for my labeled data are:**
*   **Professional:** 631 events
*   **Academic:** 303 events
*   **Personal:** 294 events

This distribution shows a significant imbalance. Professional events make up more than half of the total dataset. I will need to address this during the training phase to ensure the model does not just default to the most frequent category to get a high accuracy score.


### **Section 2: Data Ingestion and Conversion**

#### **The Engineering Reality: Bridging the Gap Between ICS and XLSX**
Turning my raw calendar data into a format that I could actually use for machine learning was a massive hurdle. I spent about two days of intense struggle trying to navigate the messy reality of the iCalendar format. On the surface, it seems like a simple file, but once you open it, you realize it is just a long stream of machine code where every event is wrapped in technical tags. A standard table has rows and columns, but this was more like a wall of text. My goal was to create a custom script that could systematically peel back these layers to extract the ten specific features I needed. Because existing tools often broke when faced with Google’s specific way of formatting these files, I had to build a custom ingestion pipeline from scratch.

#### **Solving the Folded Lines Problem**
The first major technical headache I encountered was "folded lines." In these files, if an event summary or description is too long, the system arbitrarily breaks it into multiple physical lines with a space or tab at the start of the next row. If I had just read the file line by line like a normal text document, my event titles would have been chopped in half. My model would have been training on broken and incomplete data as a result. 

To fix this, I wrote a function called `unfold_ics_lines`. This piece of code scans the entire text and checks if a line starts with a hidden space. If it does, the script realizes this is just a continuation of the line above it and joins them back together. This step was essential for making sure my primary feature, the event summary, stayed in one piece.

```python
def unfold_ics_lines(text: str) -> list[str]:
    lines = text.splitlines()
    unfolded: list[str] = []
    for line in lines:
        # If the line starts with a space or tab, it belongs to the line before it
        if (line.startswith(" ") or line.startswith("\t")) and unfolded:
            unfolded[-1] += line[1:]
        else:
            unfolded.append(line)
    return unfolded
```

#### **Isolating Events with Regular Expressions**
Once the text was unfolded and readable, I needed a way to find where each individual event started and ended. An `.ics` file is full of extra information like timezone rules and calendar settings that I did not need for my analysis. I only cared about the actual activities. I used a tool called regular expressions (regex) to build a pattern that could hunt down everything sitting between the `BEGIN:VEVENT` and `END:VEVENT` tags.

```python
def split_vevent_blocks(ics_text: str) -> list[str]:
    # This pattern hunts for the specific start and end markers of a calendar event
    pattern = re.compile(r"BEGIN:VEVENT\r?\n(.*?)\r?\nEND:VEVENT", re.DOTALL)
    return [match.group(1) for match in pattern.finditer(ics_text)]
```

This chunk allowed me to transform one giant and confusing file into a list of 1,228 individual strings. Once I had them separated, it was much easier to pick out details like the location and timezone for each one.

#### **Handling Inconsistent Lengths and Durations**
A particularly difficult issue was how Google records how long an event lasts. I found that many entries had a start and end time but were missing a specific "Duration" field. Even worse, some events were just simple reminders that had no end time at all. I had to write logic that acted as a safety net to ensure my final table did not have empty gaps. 

My script first looks for a native duration tag. If it cannot find one, it does the math itself by subtracting the start time from the end time. If there is no end time at all, it marks the event as "instant" by setting the duration to zero. This ensured my final dataset was consistent and ready for the model to process.

```python
def duration_minutes(dtstart: datetime | None, dtend: datetime | None, raw_duration: str) -> str:
    parsed = parse_ics_duration(raw_duration)
    if parsed is not None:
        return str(int(parsed.total_seconds() // 60))
    
    # If the duration field is missing, we calculate it ourselves from the timestamps
    if dtstart is not None and dtend is not None:
        return str(int((dtend - dtstart).total_seconds() // 60))
    
    return "0" # Default for instant events like reminders
```

#### **The Timezone Crash and Final Export**
The final roadblock happened at the very last second when I tried to save my work. Python’s internal clock handles timezones in a very complex "timezone-aware" way, but the standard Excel export tool crashes when it tries to save those specific objects. To solve this, I normalized all my timezone names and then used a final loop to force every single column into plain text before writing the file. This prevented the software from crashing while keeping all the technical details like UTC offsets and daylight savings info intact.

```python
# Force all values to text to prevent Excel timezone errors from crashing the script
for col in df.columns:
    df[col] = df[col].astype(str)

df.to_excel(output_path, index=False)
```

#### **Exhibit: The Structured Dataset**
By using these specific code segments, I successfully turned a chaotic machine archive into a clean and structured table. This 10-column spreadsheet contains the exact features I needed. It includes the summaries, durations, and normalized timezone data that I then used to manually label my data and train my classification models.

<div style="text-align: center;">
  <img src="Cleaned_data.png" alt="Plot" width="900">
  <p><em>Figure 2. A screenshot of my final structured data in Excel showing the clean columns for summaries and the calculated durations that I spent two days fighting to get right.</em></p>
</div>



### **Section 3: Cleaning, Pre-processing, and Exploratory Data Analysis**

#### **Data Cleaning and Pre-processing**

The first step in preparing the calendar data involved cleaning the raw records to ensure they were consistent and usable for a machine learning model. I began by handling missing values in the summary column. Any entry that lacked a title was filled with an empty string and the entire column was converted to a standard text format. This step prevents the code from failing when it encounters null values. 

I also addressed the category labels. To ensure the model did not misinterpret formatting differences, I forced all labels to a string type and stripped away any leading or trailing whitespace. This was a critical step because it prevented a category like "academic " with an extra space from being treated as a separate class from "academic" without one. I then removed any rows that lacked a label entirely so the model would only train on validated data.

Because calendar data is inherently chronological, the raw file followed a specific timeline of my life. To prevent the model from learning temporary patterns based on a specific semester or month, I performed a random shuffle of the 1,228 rows using a fixed random seed of 42. Shuffling is a vital part of the process because it ensures that the training and testing sets are representative of the entire time period rather than just one specific block of time. I used a stratified 80/20 split to create these sets. Stratification ensures that the proportions of Professional, Academic, and Personal events remain the same in both the training and testing groups, which allows for a more reliable evaluation of model performance.

#### **Feature Engineering: TF-IDF Vectorization**

The most important part of this section is the transformation of human language into numerical data that a computer can process. I used a tool called the TF-IDF Vectorizer to turn my event summaries into numeric vectors. I got this tool from the pre-class work for CS156 Session 5 (Max Likelihood 1: Naive Bayes). This is code from class that I adapted for use in my project.

This algorithm works by looking at the importance of words. Words that appear frequently in one specific summary but rarely in others are given higher weights. This is because they provide a strong signal for what that event is about. Common words like "the" or "and" are down-weighted because they do not help the model distinguish between an academic session and a professional meeting. By converting the text into these weighted scores, the model can look for patterns in the vocabulary I use across the different spheres of my life.

#### **Exploratory Data Analysis**

Before building the model, I performed a basic analysis to understand the composition of my digital archive. The final dataset contains 1,228 labeled events. There is a clear imbalance in the categories. Professional events are the most common with 631 entries. Academic events follow with 303 entries, and personal events are the smallest group with 294 entries. This distribution shows that my university account is heavily weighted toward professional and work related tasks.

I also examined the linguistic structure of the summaries using simple engineered features like word and character counts. On average, my calendar entries are very brief, consisting of only about five words per summary. Most entries are between 15 and 40 characters long. This brevity means the model must be very efficient. Since there is very little text for each event, the model will need to rely on specific, high weight keywords to make accurate predictions.

<div style="display: flex; justify-content: center; gap: 20px;">
  <img src="use_image1.png" style="width: 40%;">
  <img src="use_image2.png" style="width: 40%;">
</div>
<p style="text-align: center; font-style: italic;">
  
**Figure 3. Dataset Composition and Complexity.**
The bar chart illustrates how my professional life dominates my digital schedule. The histogram shows that the vast majority of my event summaries are quite short. This confirms that the success of the model will depend on its ability to pick out specific, meaningful terms from very limited text.
</p>


#### **Implementation and Descriptive Statistics**

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Cleaning and Shuffling 
RANDOM_SEED = 42
df = pd.read_excel("Data_Labeled.xlsx")

# Shuffle to break the chronological order of the calendar
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Clean summaries and labels
df = df[["summary", "Label"]].copy()
df["summary"] = df["summary"].fillna("").astype(str)
df["Label"] = df["Label"].astype(str).str.strip()
df = df[df["Label"] != ""]

# 2. Exploratory Data Analysis
eda_df = df.copy()
eda_df['summary_char_len'] = eda_df['summary'].str.len()
eda_df['summary_word_count'] = eda_df['summary'].str.split().str.len()

print(f"Total labeled rows: {len(eda_df)}")
print("\nEvent Distribution by Class:")
print(eda_df['Label'].value_counts())
print("\nSummary Statistics for Word Counts:")
print(eda_df['summary_word_count'].describe())

# 3. Splitting and Feature Engineering
# Adapted from CS156 Session 5 PCW
X_train_text, X_test_text, y_train, y_test = train_test_split(
    eda_df["summary"], 
    eda_df["Label"], 
    test_size=0.20, 
    random_state=RANDOM_SEED, 
    stratify=eda_df["Label"]
)

# Initialize the vectorizer as used in class
vectorizer = TfidfVectorizer(
    lowercase=True, 
    stop_words="english", 
    min_df=2
)
X_train_vec = vectorizer.fit_transform(X_train_text)

# 4. Visualizations
plt.figure(figsize=(12, 5))

# Plot the balance of classes
plt.subplot(1, 2, 1)
sns.countplot(data=eda_df, x='Label', order=eda_df['Label'].value_counts().index)
plt.title('Distribution of Life Spheres')

# Plot the length of event summaries
plt.subplot(1, 2, 2)
sns.histplot(data=eda_df, x='summary_char_len', bins=30, kde=True)
plt.title('Distribution of Summary Lengths')
plt.xlabel('Character Count')

plt.tight_layout()
plt.show()
```





### **Section 4: Analysis Strategy and Data Partitioning**

After completing data selection, ingestion, and preprocessing in the first three sections, the project enters the formal modeling phase. The analytical objective is to build a supervised machine learning system that can classify each calendar event into one of three categories: **Academic**, **Professional**, or **Personal**. Although one of the selected algorithms is called Logistic Regression, the task is not regression in the sense of predicting a continuous value. Instead, this is a multiclass classification problem where each event receives a single categorical label. Framing this distinction clearly is important because it determines the loss behavior, evaluation metrics, and interpretation strategy used throughout the rest of the notebook.

At a mathematical level, the learning target is the conditional probability of a class given an event summary representation, written as $P(y=k\mid x)$. Here, $x$ denotes the transformed text features for one event summary, and $k$ is one of the three label values. The model should learn decision boundaries that generalize to unseen events, not merely memorize the training examples. That requirement is especially relevant in this project because event summaries are short and repetitive. Many examples share common words such as ?meeting,? ?session,? and ?call,? so model quality depends on learning subtle contextual differences rather than relying on one-word shortcuts.

A careful split protocol is therefore central to credible results. The full labeled dataset contains 1,228 events, with visible class imbalance: Professional events are most frequent, while Academic and Personal classes are smaller and closer to each other in size. If I were to use a naive random split without constraints, some runs could produce a test set with distorted class proportions, making performance comparisons unstable. To avoid that issue, I use a **stratified** 80/20 partition. This produces 982 training examples and 246 test examples while preserving the class distribution structure from the full dataset. Stratification ensures that each model is exposed to representative class frequencies during training and is judged on a representative test distribution.

I also shuffle before splitting because calendar data is naturally time-ordered. In raw form, earlier years and recent years are clustered in sequence, and this chronology can unintentionally leak period-specific language into train or test subsets. For example, specific course codes, project names, or internship cycles may be concentrated in one calendar window. Without shuffling, the model might overfit to temporal artifacts rather than learning general textual class structure. Shuffling first and then splitting reduces this risk. I fix the random seed to 42 to ensure full reproducibility across reruns and across model families.

The same partitioning scheme is reused for Naive Bayes and Logistic Regression. This is non-negotiable for fair comparison. If model A and model B are evaluated on different test samples, differences in scores can be caused by sample difficulty rather than model capability. Holding the split constant means metric differences are attributable to model behavior.

Finally, the split strategy directly supports assignment requirements around out-of-sample evaluation. Training data is used for parameter fitting and tuning workflows, while test data remains the holdout reference for generalization performance. This separation is what makes the final conclusions defensible.

#### **Exhibit: Partitioning Code**
```python
from sklearn.model_selection import train_test_split

X_text = df["summary"]
y = df["Label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
```


### **Section 5: Model Selection and Mathematical Underpinnings**

This section explains why Multinomial Naive Bayes and Multinomial Logistic Regression were selected, how each algorithm works, and why both are appropriate for sparse text classification. The project uses event summaries as the primary predictive signal, and summary text is transformed using TF-IDF so that models can consume numeric inputs. TF-IDF is especially suitable here because calendar titles are brief and keyword-driven. It reduces the influence of common language and amplifies terms that carry class-specific meaning.

#### **Naive Bayes as a Generative Baseline**
Multinomial Naive Bayes models the posterior class probability through class priors and per-feature likelihoods. In practice, it applies Bayes logic with a conditional independence assumption among features given the class. The general decision form is:

$$
P(y \mid x_1, x_2, \dots, x_n) \propto P(y)\prod_{i=1}^{n} P(x_i\mid y)
$$

In text settings, this assumption is simplification-heavy because words in phrases are not truly independent. However, despite its naive assumption, the model often performs strongly due to its stability and low variance, especially when input features are sparse and high dimensional.

One critical component is smoothing. Without smoothing, unseen tokens in a class can force class likelihood terms to zero and collapse probabilities. Additive smoothing with parameter $\alpha$ addresses this:

$$
P(x_i\mid c)=\frac{\mathrm{count}(x_i,c)+\alpha}{\sum_x \mathrm{count}(x,c)+\alpha |V|}
$$

where $|V|$ is the vocabulary size. In this project, tuning shows that smaller smoothing performs better because class-signature tokens are informative and should not be over-smoothed.

#### **Logistic Regression as a Discriminative Comparator**
Multinomial Logistic Regression models class boundaries directly rather than modeling token generation per class. It computes one linear score per class:

$$
z_k=\beta_{k,0}+\beta_{k,1}x_1+\cdots+\beta_{k,n}x_n
$$

and transforms these scores into probabilities via softmax:

$$
P(y=k\mid x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

Training optimizes parameters by maximizing log-likelihood (equivalently minimizing cross-entropy):

$$
\log L(\beta)=\sum_{i=1}^{N}\log P(y_i\mid x_i;\beta)
$$

In practical terms, Logistic Regression provides interpretable coefficients and often superior class separation on linear text boundaries. It also includes explicit regularization control through $C$, enabling a structured bias-variance tradeoff.

#### **Why This Pair Is Methodologically Strong**
Using both models is not redundancy; it is comparative design. Naive Bayes offers a probabilistic, low-complexity baseline that is fast and robust. Logistic Regression offers a stronger discriminative mechanism with calibrated regularization control. Agreement between both models increases confidence in dataset signal quality; divergence helps reveal where assumptions matter.

Given the observed class imbalance, Logistic Regression uses `class_weight="balanced"` so minority class errors are penalized more proportionally during optimization. This choice aligns with Macro-F1-focused evaluation.

#### **Exhibit: Model Initialization**
```python
nb_model = MyMultinomialNB(alpha=1.0)

lr_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42
)
```



### **Section 6: Model Training and Hyperparameter Tuning**

With both model families defined, I moved from baseline fitting to systematic hyperparameter tuning. This phase is critical because default settings can hide model capacity or produce misleading underperformance. I used **Macro-F1** as the primary selection metric throughout tuning because the class distribution is imbalanced and Macro-F1 forces equal attention to each class.

#### **Naive Bayes Tuning Strategy**
For Multinomial Naive Bayes, the central hyperparameter is smoothing strength $\alpha$. I evaluated the following grid:

$$
\alpha \in \{0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0\}
$$

I used two complementary views:
1. A summary curve plotting Macro-F1 and accuracy across alpha values.
2. A confusion-matrix grid to inspect class-level error movement.

The quantitative trend was consistent: increasing $\alpha$ reduced performance. This indicates that strong smoothing blurred useful class-specific token patterns in the summaries. In practical terms, terms that strongly indicate one class were dampened toward uniformity, making classes less separable. The best value, $\alpha=0.01$, preserved sharper lexical signals and produced the strongest Macro-F1.

<div style="text-align: center;">
  <img src="nb_alpha_tuning.png" alt="Naive Bayes alpha tuning" width="680">
  <p><em>Figure 4. Naive Bayes alpha sweep summary (Accuracy and Macro-F1).</em></p>
</div>

<div style="text-align: center;">
  <img src="nb_alpha_grid_3x3.png" alt="Naive Bayes alpha 3x3 grid" width="1020">
  <p><em>Figure 5. Naive Bayes alpha comparison as a 3x3 confusion-matrix grid (single seed).</em></p>
</div>

The 3x3 grid is particularly useful because it reveals where performance differences come from. It is not only that the scalar metric declines with larger alpha; the confusion patterns also become less class-specific. In several high-alpha cells, minority-class recall weakens because the model becomes overly conservative and less responsive to rare informative tokens.

#### **Logistic Regression Tuning Strategy**
For Logistic Regression, the key parameter is regularization strength via $C$ (inverse regularization). I evaluated:

$$
C \in \{0.01, 0.1, 0.5, 1, 2, 5, 10, 20, 50, 100, 200\}
$$

As $C$ increased, performance improved and then stabilized in a high-performing plateau. This behavior is coherent with sparse text: too much regularization suppresses meaningful coefficients, while moderate-to-high $C$ allows the model to emphasize discriminative terms. I selected $C=20$ as a practical choice from this top plateau, balancing strong performance and stability.

<div style="text-align: center;">
  <img src="lr_c_tuning.png" alt="Logistic regression C tuning" width="680">
  <p><em>Figure 6. Logistic Regression C sweep (log-scale x-axis).</em></p>
</div>

#### **Why This Tuning Phase Matters**
This phase does more than optimize scores. It helps diagnose the structure of the dataset itself. Naive Bayes favoring very low smoothing implies high lexical specificity in event summaries. Logistic Regression favoring weaker regularization implies that allowing stronger token coefficients improves class discrimination. Together, these observations characterize the signal quality in the archive.

#### **Implementation Discipline**
All tuning runs were performed with consistent preprocessing and shared split logic. This prevented accidental performance inflation from changing vectorizer behavior, token filters, or split definitions between runs. In other words, hyperparameter differences not pipeline drift explain score differences.

By the end of this section, each model has a tuned configuration based on class-balanced criteria:
- Naive Bayes: $\alpha=0.01$
- Logistic Regression: $C=20$

These tuned models are carried forward into out-of-sample evaluation.

#### **Additional Diagnostic Interpretation**
An additional reason this tuning behavior is meaningful is that both models were exposed to the same TF-IDF geometry, yet their optimal hyperparameters point to different biases. Naive Bayes prefers very low smoothing, meaning token-class associations should remain sharp and local. Logistic Regression prefers weaker regularization than very small $C$ values, meaning useful coefficients should not be heavily shrunk toward zero. Taken together, the two curves suggest that this dataset has strong lexical anchors but still benefits from flexible class-boundary weighting.

This diagnostic consistency strengthens confidence in the selected hyperparameters. The selected values are not random peaks in noisy searches; they align with how short schedule text behaves mathematically. In practical terms, frequent class-coded terms should remain influential, while excessive smoothing or excessive shrinkage would remove exactly the signal the models need.


### **Section 7: Out-of-Sample Predictions and Performance Metrics**

After hyperparameter selection, I evaluated final models on the held-out test set of 246 events. This set was never used for fitting model parameters, and therefore serves as the generalization benchmark. The purpose of this section is to answer a narrow question: how well do the tuned models classify unseen summaries from the same archive distribution?

Final model settings were:
- Naive Bayes: $\alpha=0.01$
- Logistic Regression: $C=20$

Shared-split test metrics:
- **Naive Bayes:** Accuracy = **0.9106**, Macro-F1 = **0.9054**
- **Logistic Regression:** Accuracy = **0.9228**, Macro-F1 = **0.9170**

These results show strong performance from both models. Importantly, the ranking is consistent across both global correctness (accuracy) and class-balanced quality (Macro-F1), with Logistic Regression performing slightly better.

Why this consistency matters: if only accuracy improved while Macro-F1 remained flat, the improvement might be mostly majority-class gain. Here, Macro-F1 also improves, indicating better balance across Academic, Professional, and Personal classes.

<div style="text-align: center;">
  <img src="model_metric_comparison.png" alt="Best model metric comparison" width="720">
  <p><em>Figure 7. Best-model metric comparison on the same holdout split.</em></p>
</div>

The metric comparison confirms that the final selection is not arbitrary. Logistic Regression provides a measurable edge while Naive Bayes remains a competitive baseline. This closeness is informative: it suggests that class signal in the summaries is strong enough for both probabilistic and linear-discriminative methods.

To move beyond scalar metrics, I include class-structure diagnostics.

<div style="text-align: center;">
  <img src="lr_confusion_test.png" alt="Logistic confusion matrix" width="600">
  <p><em>Figure 8. Logistic Regression confusion matrix (test set).</em></p>
</div>

<div style="text-align: center;">
  <img src="nb_per_class_metrics.png" alt="Naive Bayes class-wise metrics" width="780">
  <p><em>Figure 9. Naive Bayes per-class precision, recall, and F1 (test set).</em></p>
</div>

These visuals provide complementary insights. The confusion matrix identifies directional errors (which classes get confused with which), while per-class bars show whether error is precision-driven, recall-driven, or both. In this dataset, Personal events are comparatively more difficult because summaries can be short and semantically broad. That pattern is expected from the data-generation process itself, not just from model weakness.

Another key interpretation point is that both models exceed 0.90 on the primary metrics despite short text inputs. This indicates that event titles carry stable and repeated linguistic cues across years. It also validates the preprocessing and labeling strategy used in Sections 1 and 3.

From a pipeline perspective, this section fulfills the assignment requirement for out-of-sample prediction and metric reporting. It demonstrates that the tuned models are not only optimized in theory but also effective in practical holdout conditions.

The next section consolidates these findings visually and translates the metric-level results into final project conclusions and limitations.

#### **Evaluation Discipline and Interpretation Boundary**
It is also important to define what these metrics do and do not claim. The reported values describe model performance on held-out events drawn from the same archive distribution. They do not claim universal transfer to all calendar users or all institutions. However, within this project scope, the holdout results are strong enough to support model validity.

I also interpret the differences cautiously. A gain of roughly one to two Macro-F1 points is meaningful when comparing two tuned models on the same split, especially when both are already in a high-performance regime. At the same time, the closeness of results indicates that dataset quality and labeling consistency are major drivers of success, not only model complexity.

For grading and reproducibility, this section therefore serves as the objective anchor: same split, tuned settings, explicit metrics, and visual diagnostics that connect numeric output to concrete class-level behavior.


### **Section 8: Visualizations and Final Conclusions**

The workflow has moved through ingestion, cleaning, feature engineering, model selection, tuning, and out-of-sample evaluation. The remaining task is to interpret what the results mean, where they are strongest, and what limitations remain.

The central comparison visualization is shown below.

<div style="text-align: center;">
  <img src="model_confusion_comparison.png" alt="Final confusion comparison" width="940">
  <p><em>Figure 10. Best-model confusion matrices on a shared split: Logistic Regression (C=20) vs Naive Bayes (alpha=0.01).</em></p>
</div>

The figure confirms the same ranking seen in the scalar metrics: both models perform well, and Logistic Regression is slightly stronger. The difference is meaningful but not dramatic, which is an important finding in itself. It indicates that the underlying textual signal is robust and not dependent on one specific algorithmic assumption.

#### **What the Models Learned**
Both models successfully capture domain-specific vocabulary patterns linked to the three life spheres. Academic events contain recurring course and assignment language, professional events contain recurring interview/workshop/career terms, and personal events contain a different lexical profile. This supports the project hypothesis that calendar summaries encode meaningful behavioral information despite their brevity.

### **Section 9: Executive Summary**

#### **Why Logistic Regression Wins Slightly**
Logistic Regression likely benefits from directly learning class boundaries with class-specific linear coefficients and softmax normalization. Naive Bayes remains strong but is constrained by conditional independence assumptions. In settings where token interactions and correlated wording matter, Logistic Regression can separate classes more effectively.

#### **Why Naive Bayes Still Matters**
Although not the top performer, Naive Bayes remains valuable as a baseline. It is computationally simple, highly interpretable, and performed competitively on this dataset. Keeping it in the analysis strengthens methodological rigor by demonstrating that the final conclusion is comparative, not model-cherry-picked.

#### **Primary Limitation**
The largest limitation is sampling bias in the source archive. The university-linked account over-represents formal commitments and under-represents informal personal behavior. As a result, the model is best interpreted as a classifier of recorded scheduling behavior, not a complete portrait of all lived activity.

A secondary limitation is summary brevity. Short entries reduce contextual richness and can increase ambiguity, especially for personal events where wording is less standardized.

#### **Practical and Analytical Takeaway**
Despite these limitations, the pipeline is successful. It demonstrates that raw calendar archives can be converted into reproducible, interpretable, and high-performing classifiers with careful engineering and evaluation controls. The results show that even small text fields can encode stable life-structure patterns over time.

From a broader machine learning perspective, this project illustrates the value of combining:
- strong data engineering,
- transparent baseline modeling,
- metric choices aligned with imbalance,
- and visualization-driven error diagnosis.

Together, these elements produce conclusions that are both technically grounded and substantively meaningful.

This concludes the model-analysis portion of the report and provides a defensible basis for the final executive summary and references sections.

#### **Future Improvements**
A next iteration could improve both representativeness and model performance by adding richer contextual features beyond summary text. Candidate features include event start hour, day-of-week, duration bins, and structured location indicators. These signals may help disambiguate cases where text alone is generic (for example, brief entries like ?meeting? or ?dinner?).

Another improvement is expanding the archive beyond a single account context or manually enriching underrepresented personal entries. That would reduce sampling bias and produce a more balanced reflection of actual life allocation. Finally, nested validation for hyperparameter selection and repeated-split reporting for both final models would provide even tighter statistical confidence bounds.

Even with these possible upgrades, the current results already demonstrate the central claim: a careful end-to-end machine learning workflow can extract reliable and interpretable behavioral patterns from a personal calendar archive.


### **Section 9: Executive Summary**

This project built a complete end-to-end machine learning pipeline from a personal Google Calendar archive. The workflow began with exporting raw `.ics` data, engineering a robust ingestion script to handle folded lines, timezone formatting, and missing duration values, and producing a structured `.xlsx` table suitable for modeling. After manual labeling into Academic, Professional, and Personal classes, I transformed event summaries into TF-IDF features and trained two supervised classifiers: Multinomial Naive Bayes and Multinomial Logistic Regression.

Both models performed strongly on a stratified holdout split, with Logistic Regression achieving the best overall results. Hyperparameter tuning showed that Naive Bayes performed best with very low smoothing (`alpha=0.01`) and Logistic Regression performed best on a high-performing regularization plateau (`C=20`). Confusion-matrix and class-wise metric analysis confirmed that the Personal class remained the most difficult due to shorter and less specific summaries, but performance remained high overall.

The core conclusion is that even short calendar summaries contain meaningful statistical structure that can classify life-domain activity with high reliability. The strongest limitation is archive bias: this university-linked account over-represents formal academic/professional commitments relative to informal personal life.


### **Section 10: References**

1. **Scikit-learn Documentation** (classification models, metrics, train/test splitting, TF-IDF):
   - https://scikit-learn.org/stable/
   - https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
   - https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
   - https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html

2. **Pandas Documentation** (data ingestion, cleaning, Excel export):
   - https://pandas.pydata.org/docs/

3. **Matplotlib and Seaborn Documentation** (visualization):
   - https://matplotlib.org/stable/
   - https://seaborn.pydata.org/

4. **Google Calendar Export Guidance** (source data export process):
   - https://support.google.com/calendar/answer/37111

5. **iCalendar (RFC 5545) Specification** (format behavior such as folded lines and VEVENT structure):
   - https://datatracker.ietf.org/doc/html/rfc5545

6. **Course Materials**:
   - CS156 Session 5 pre-class materials (Naive Bayes and TF-IDF foundations)
   - CS156 Session 6?9 class notes and workbook exercises (log-likelihood, logistic regression, metrics, validation)

7. **Code Assistance and Debugging Support**:
   - ChatGPT/Codex-assisted debugging and implementation support for ICS ingestion, model tuning workflows, and report structuring.
